
# Mercadona API Client

This notebook collects the Mercadona product catalogue directly from its JSON API.

## Collection process

1. Resolves the warehouse associated with the configured postal code.
2. Retrieves the product category tree.
3. Downloads products from each subcategory with retries.
4. Extracts product identifiers, prices, package information, categories, and timestamps.
5. Validates collection completeness.
6. Saves the current RAW catalogue and a dated historical RAW snapshot.

## Outputs

- Current RAW: `data/raw/mercadona_api_products_raw.csv`
- Historical RAW: `data/history/mercadona/YYYY-MM-DD_raw.csv`

Product matching, standardization, classification, and historical price processing are handled by the central pipeline.

## Collection policy

- Only complete category collections are saved.
- Existing historical snapshots are never overwritten.
- TEST_MODE allows testing selected subcategories without saving files.


In [ ]:
import re
import time
import unicodedata
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests

BASE_URL = "https://tienda.mercadona.es"
POSTAL_CODE = "28911"
LANG = "es"

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------
TEST_MODE = False
TEST_MAX_CATEGORIES = 3   # in test mode, only the first 3 subcategories

REQUEST_PAUSE = 1.0       # seconds between requests
MAX_RETRIES = 3
TIMEOUT = 20

HEADERS = {
    "User-Agent": "UC3M-WebAnalytics-StudentProject/1.0 (academic, non-commercial)",
    "Accept": "application/json",
}

In [2]:
# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------


def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if (
            (path / "api_clients").is_dir()
            and (path / "src").is_dir()
            and (path / "data").is_dir()
        ):
            return path

    raise FileNotFoundError(
        "Could not locate the project root."
    )



BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
HISTORY_DIR = BASE_DIR / "data" / "history" / "mercadona"

# Keep the filename expected by the Mercadona API adapter.
RAW_PATH = RAW_DIR / "mercadona_api_products_raw.csv"

RUN_DATE = datetime.now().strftime("%Y-%m-%d")
RAW_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_raw.csv"

print("Project:", BASE_DIR)
print("Collection date:", RUN_DATE)
print("Current RAW:", RAW_PATH)
print("Historical RAW:", RAW_HISTORY_PATH)

Project: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-20
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\mercadona_api_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\mercadona\2026-11-20_raw.csv


## API requests

`request_json` makes one request with retries (exponential backoff on network errors,
429 or 5xx) and pauses after every call. `resolve_warehouse` asks the API which
warehouse serves our postal code, since prices and stock are tied to it.

In [3]:
session = requests.Session()
session.headers.update(HEADERS)


def request_json(method, path, **kwargs):
    url = BASE_URL + path

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.request(method, url, timeout=TIMEOUT, **kwargs)

            if response.status_code == 429 or response.status_code >= 500:
                raise requests.HTTPError(f"HTTP {response.status_code}")

            response.raise_for_status()
            time.sleep(REQUEST_PAUSE)
            return response

        except requests.RequestException as exc:
            if attempt == MAX_RETRIES:
                raise
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({exc}). Retrying in {wait} s...")
            time.sleep(wait)


def resolve_warehouse(postal_code):
    response = request_json(
        "PUT",
        "/api/postal-codes/actions/change-pc/",
        json={"new_postal_code": postal_code},
    )
    warehouse = response.headers.get("x-customer-wh")
    if not warehouse:
        raise RuntimeError("The API did not return a warehouse for this postal code.")
    return warehouse


WAREHOUSE = resolve_warehouse(POSTAL_CODE)
print(f"Postal code {POSTAL_CODE} -> warehouse {WAREHOUSE}")

Postal code 28911 -> warehouse mad1


## Category tree

The tree has two levels: ~26 top-level categories (e.g. *Aceite, especias y salsas*)
containing ~150 subcategories (e.g. *Aceite de oliva*). Products are only listed at
subcategory level, grouped into sections.

In [4]:
def get_subcategories():
    data = request_json(
        "GET", "/api/categories/", params={"lang": LANG, "wh": WAREHOUSE}
    ).json()

    subcategories = []
    for top in data["results"]:
        for sub in top["categories"]:
            subcategories.append({
                "top_category": top["name"],
                "subcategory_id": sub["id"],
                "subcategory": sub["name"],
            })
    return subcategories


subcategories = get_subcategories()

if TEST_MODE:
    subcategories = subcategories[:TEST_MAX_CATEGORIES]

print("Subcategories to fetch:", len(subcategories))

Subcategories to fetch: 3


## Flatten the JSON

The useful fields are in `price_instructions`:

- `unit_price`: price of the item as sold (one bottle, one pack...).
- `reference_price` + `reference_format`: price per reference unit (`kg`, `L`,
  `100 g`, `100 ml`, `ud`, `dc`/`dz` = dozen). This is the price-per-unit the
  website shows.
- `unit_size` + `size_format`: total quantity of the item (`kg`, `l` or `ud`).
- `is_pack`, `total_units`, `pack_size`, `unit_name`: pack composition
  (e.g. 12 mini bottles x 0.1 kg).
- `previous_unit_price`: the price before a recent price change, if any.

In [5]:
def describe_quantity(pi):
    """Builds a readable quantity string, e.g. '12 mini botellas x 0.1 kg' or '1.5 l'."""
    size_format = pi.get("size_format") or ""

    if pi.get("is_pack") and pi.get("total_units") and pi.get("pack_size"):
        unit_name = pi.get("unit_name") or "ud."
        return f"{pi['total_units']} {unit_name} x {pi['pack_size']} {size_format}"

    if pi.get("unit_size") is not None:
        return f"{pi['unit_size']} {size_format}"

    return None


def flatten_product(product, subcategory, section, scraped_at):
    pi = product.get("price_instructions") or {}

    return {
        "supermarket": "Mercadona",
        "product_id": product.get("id"),
        "name": product.get("display_name"),
        "packaging": product.get("packaging"),
        "top_category": subcategory["top_category"],
        "category": subcategory["subcategory"],
        "section": section,
        "unit_price": pi.get("unit_price"),
        "previous_unit_price": pi.get("previous_unit_price"),
        "price_decreased": pi.get("price_decreased"),
        "reference_price": pi.get("reference_price"),
        "reference_format": pi.get("reference_format"),
        "bulk_price": pi.get("bulk_price"),
        "unit_size": pi.get("unit_size"),
        "size_format": pi.get("size_format"),
        "is_pack": pi.get("is_pack"),
        "total_units": pi.get("total_units"),
        "pack_size": pi.get("pack_size"),
        "unit_name": pi.get("unit_name"),
        "approx_size": pi.get("approx_size"),
        "quantity_raw": describe_quantity(pi),
        "url": product.get("share_url"),
        "image_url": product.get("thumbnail"),
        "postal_code": POSTAL_CODE,
        "warehouse": WAREHOUSE,
        "scraped_at": scraped_at,
    }

## Download the catalogue

If a subcategory still fails after the retries, it is logged as `ERROR` in the
summary and the loop moves on to the next one.

In [6]:

# --------------------------------------------------
# COLLECT MERCADONA CATALOGUE WITH DEFERRED RETRIES
# --------------------------------------------------

# Initial pass + two additional retry passes.
MAX_ADDITIONAL_ATTEMPTS = 2
MAX_COLLECTION_PASSES = 1 + MAX_ADDITIONAL_ATTEMPTS

# Store successful subcategories separately to prevent duplicates.
successful_results = {}

# Store one final summary record per subcategory.
summary_records = {}


# --------------------------------------------------
# COLLECTION PASSES
# --------------------------------------------------

for pass_number in range(1, MAX_COLLECTION_PASSES + 1):

    # Only collect subcategories that have not succeeded.
    pending_indices = [
        index
        for index in range(len(subcategories))
        if index not in successful_results
    ]

    if not pending_indices:
        print("\nAll subcategories collected successfully.")
        break

    print("\n" + "=" * 70)
    print(
        f"COLLECTION PASS "
        f"{pass_number}/{MAX_COLLECTION_PASSES}"
    )
    print("Pending subcategories:", len(pending_indices))
    print("=" * 70)

    for position, index in enumerate(pending_indices, start=1):

        subcategory = subcategories[index]

        print(
            f"\n[{position}/{len(pending_indices)}] "
            f"{subcategory['top_category']} > "
            f"{subcategory['subcategory']} "
            f"(attempt {pass_number})"
        )

        started = time.monotonic()

        status = "ERROR"
        error_message = None
        category_rows = []

        try:

            # Restart the complete subcategory request.
            response = request_json(
                "GET",
                (
                    f"/api/categories/"
                    f"{subcategory['subcategory_id']}/"
                ),
                params={
                    "lang": LANG,
                    "wh": WAREHOUSE
                }
            )

            data = response.json()

            # Validate the expected API response structure.
            if (
                not isinstance(data, dict)
                or not isinstance(data.get("categories"), list)
            ):
                raise ValueError(
                    "Invalid category response structure."
                )

            scraped_at = datetime.now().isoformat(
                timespec="seconds"
            )

            # Build a temporary list for this attempt.
            # Partial results are never added to the final RAW.
            for section in data["categories"]:

                products = section.get("products", [])

                if not isinstance(products, list):
                    raise ValueError(
                        "Invalid products list in API response."
                    )

                for product in products:

                    category_rows.append(
                        flatten_product(
                            product,
                            subcategory,
                            section["name"],
                            scraped_at
                        )
                    )

            # The subcategory was processed successfully.
            successful_results[index] = category_rows

            status = "OK"

            print(
                f"SUCCESS: {len(category_rows)} products "
                f"collected."
            )

        except Exception as exc:

            error_message = (
                f"{type(exc).__name__}: {exc}"
            )

            print(f"FAILED: {error_message}")
            print("Continuing with the next subcategory.")

            # Discard products from the failed attempt.
            category_rows = []

        elapsed = round(
            time.monotonic() - started,
            2
        )

        previous_seconds = summary_records.get(
            index, {}
        ).get("seconds", 0)

        # Replace the previous status for this subcategory.
        summary_records[index] = {
            **subcategory,
            "products": len(category_rows),
            "status": status,
            "attempts": pass_number,
            "seconds": round(
                previous_seconds + elapsed,
                2
            ),
            "error": error_message,
        }

    # --------------------------------------------------
    # PASS SUMMARY
    # --------------------------------------------------

    remaining_indices = [
        index
        for index in range(len(subcategories))
        if index not in successful_results
    ]

    print("\nPASS SUMMARY")
    print(
        "Successful subcategories:",
        len(successful_results)
    )
    print(
        "Remaining failures:",
        len(remaining_indices)
    )

    if (
        remaining_indices
        and pass_number < MAX_COLLECTION_PASSES
    ):

        print("\nSubcategories to retry:")

        for index in remaining_indices:

            subcategory = subcategories[index]

            print(
                f"  - {subcategory['top_category']} > "
                f"{subcategory['subcategory']}"
            )

        # Pause before retrying failed API requests.
        time.sleep(10)


# --------------------------------------------------
# BUILD FINAL RAW ROWS
# --------------------------------------------------

# Preserve the original subcategory order.
# Each successful subcategory is included exactly once.

rows = []

for index in range(len(subcategories)):

    if index in successful_results:
        rows.extend(successful_results[index])


# --------------------------------------------------
# BUILD FINAL SUMMARY
# --------------------------------------------------

# Exactly one summary record per subcategory.
summary = [
    summary_records[index]
    for index in range(len(subcategories))
]

df_summary = pd.DataFrame(summary)


# --------------------------------------------------
# FINAL COLLECTION REPORT
# --------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MERCADONA COLLECTION SUMMARY")
print("=" * 70)

print("Total subcategories:", len(subcategories))

successful_count = int(
    df_summary["status"].eq("OK").sum()
)

failed_count = int(
    df_summary["status"].ne("OK").sum()
)

recovered = df_summary[
    df_summary["status"].eq("OK")
    & df_summary["attempts"].gt(1)
]

failed = df_summary[
    df_summary["status"].ne("OK")
]

print("Successful subcategories:", successful_count)
print("Recovered after retry:", len(recovered))
print("Final failures:", failed_count)
print("Total RAW rows:", len(rows))

print("\nFinal status distribution:")
print(df_summary["status"].value_counts())

if not recovered.empty:

    print("\nRECOVERED SUBCATEGORIES:")

    display(
        recovered[
            [
                "top_category",
                "subcategory",
                "products",
                "attempts"
            ]
        ]
    )

if not failed.empty:

    print("\nWARNING: Some subcategories remain incomplete.")

    display(
        failed[
            [
                "top_category",
                "subcategory",
                "status",
                "attempts",
                "error"
            ]
        ]
    )

    print(
        "Incomplete data will not be published."
    )

else:

    print("\nAll subcategories collected successfully.")

    print(
        "Catalogue is ready for validation and saving."
    )



COLLECTION PASS 1/3
Pending subcategories: 3

[1/3] Aceite, especias y salsas > Aceite, vinagre y sal (attempt 1)
SUCCESS: 36 products collected.

[2/3] Aceite, especias y salsas > Especias (attempt 1)
SUCCESS: 49 products collected.

[3/3] Aceite, especias y salsas > Mayonesa, ketchup y mostaza (attempt 1)
SUCCESS: 19 products collected.

PASS SUMMARY
Successful subcategories: 3
Remaining failures: 0

All subcategories collected successfully.

FINAL MERCADONA COLLECTION SUMMARY
Total subcategories: 3
Successful subcategories: 3
Recovered after retry: 0
Final failures: 0
Total RAW rows: 104

Final status distribution:
status
OK    3
Name: count, dtype: int64

All subcategories collected successfully.
Catalogue is ready for validation and saving.


In [7]:
# Prepare raw dataframe
df_raw = pd.DataFrame(rows)

if df_raw.empty:
    raise RuntimeError("No products were collected.")

# Preserve original API data before adding search-term columns
df_raw_original = df_raw.copy()

print("Raw rows:", len(df_raw))
print("Distinct product IDs:", df_raw["product_id"].nunique())
print("Raw data ready for processing.")


Raw rows: 104
Distinct product IDs: 103
Raw data ready for processing.


In [8]:
# --------------------------------------------------
# VALIDATE AND SAVE MERCADONA RAW DATA
# --------------------------------------------------

import os
import tempfile

if TEST_MODE:
    print("TEST MODE: no files will be saved.")

else:
    # --------------------------------------------------
    # VALIDATE COLLECTION
    # --------------------------------------------------

    if df_raw_original.empty:
        raise RuntimeError(
            "No products collected. No files saved."
        )

    if df_summary.empty:
        raise RuntimeError(
            "No category results recorded. No files saved."
        )

    if (
        len(df_summary) != len(subcategories)
        or not df_summary["status"].eq("OK").all()
    ):
        raise RuntimeError(
            "Incomplete category collection. No files saved."
        )

    required_columns = [
        "product_id",
        "name",
        "unit_price",
        "reference_price",
        "reference_format",
        "scraped_at",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df_raw_original.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing RAW columns: {missing_columns}"
        )

    # Validate names and selling prices without changing RAW values.
    valid_names = (
        df_raw_original["name"].notna()
        & df_raw_original["name"].astype(str).str.strip().ne("")
    )

    numeric_prices = pd.to_numeric(
        df_raw_original["unit_price"],
        errors="coerce"
    )

    valid_prices = (
        numeric_prices.notna()
        & numeric_prices.gt(0)
    )

    if not (valid_names & valid_prices).any():
        raise RuntimeError(
            "No products with valid names and prices. "
            "No files saved."
        )

    invalid_rows = int(
        (~(valid_names & valid_prices)).sum()
    )

    if invalid_rows:
        print(
            f"WARNING: {invalid_rows} RAW rows have "
            "missing names or invalid prices."
        )
        print("These rows will be preserved.")

    # --------------------------------------------------
    # PROTECT HISTORICAL SNAPSHOTS
    # --------------------------------------------------

    if RAW_HISTORY_PATH.exists():
        raise FileExistsError(
            f"Historical RAW already exists: {RAW_HISTORY_PATH}. "
            "No files were modified."
        )

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------
    # PREPARE TEMPORARY RAW FILES
    # --------------------------------------------------

    def prepare_raw_temp(destination):
        fd, temp_name = tempfile.mkstemp(
            prefix=".mercadona_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:
            with os.fdopen(
                fd,
                "w",
                encoding="utf-8-sig",
                newline=""
            ) as file:
                df_raw_original.to_csv(file, index=False)

            return Path(temp_name)

        except Exception:
            Path(temp_name).unlink(missing_ok=True)
            raise

    history_temp = None
    current_temp = None

    try:
        history_temp = prepare_raw_temp(RAW_HISTORY_PATH)
        current_temp = prepare_raw_temp(RAW_PATH)

        # Create history without overwriting an existing snapshot.
        with history_temp.open("rb") as source:
            with RAW_HISTORY_PATH.open("xb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

        print("Saved historical RAW:", RAW_HISTORY_PATH)

        # Update current RAW after historical archiving succeeds.
        os.replace(current_temp, RAW_PATH)

        print("Saved current RAW:", RAW_PATH)

    finally:
        if history_temp is not None:
            history_temp.unlink(missing_ok=True)

        if current_temp is not None:
            current_temp.unlink(missing_ok=True)

    # --------------------------------------------------
    # COLLECTION SUMMARY
    # --------------------------------------------------

    print("\nCOLLECTION SUMMARY")
    print("Date:", RUN_DATE)
    print("Status: COMPLETE")
    print("RAW rows:", len(df_raw_original))
    print(
        "Distinct product IDs:",
        df_raw_original["product_id"].nunique()
    )
    print("Categories collected:", len(df_summary))
    print("Rows requiring validation:", invalid_rows)

    print("\nSaved outputs:")
    print(RAW_PATH)
    print(RAW_HISTORY_PATH)

TEST MODE: no files will be saved.
